In [2]:
import csv
import re
import time
from urllib.parse import urljoin, urlparse, urlunparse

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter, Retry

BASE = "https://f-droid.org"
HEADERS = {"User-Agent": "Mozilla/5.0"}

session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=0.6,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
session.mount("https://", HTTPAdapter(max_retries=retries))
session.headers.update(HEADERS)

def get_soup(url, timeout=20):
    resp = session.get(url, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


In [3]:
def get_category_links_from_home():
    """从 https://f-droid.org/en/packages/ 读取所有分类入口链接"""
    url = urljoin(BASE, "/en/packages/")
    soup = get_soup(url)
    post_content = soup.find("div", class_="post-content")
    links = []
    if not post_content:
        raise RuntimeError("未找到 post-content 区块")

    for a in post_content.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/en/categories/"):
            links.append(urljoin(BASE, href))

    return sorted(set(links))

category_links = get_category_links_from_home()
print(f"发现 {len(category_links)} 个分类链接")
for l in category_links[:]:
    print(l)


发现 56 个分类链接
https://f-droid.org/en/categories/app-store-updater/
https://f-droid.org/en/categories/bookmark/
https://f-droid.org/en/categories/browser/
https://f-droid.org/en/categories/calculator/
https://f-droid.org/en/categories/calendar-agenda/
https://f-droid.org/en/categories/cloud-storage-file-sync/
https://f-droid.org/en/categories/connectivity/
https://f-droid.org/en/categories/development/
https://f-droid.org/en/categories/dns-hosts/
https://f-droid.org/en/categories/draw/
https://f-droid.org/en/categories/ebook-reader/
https://f-droid.org/en/categories/email/
https://f-droid.org/en/categories/file-encryption-vault/
https://f-droid.org/en/categories/file-transfer/
https://f-droid.org/en/categories/finance-manager/
https://f-droid.org/en/categories/gallery/
https://f-droid.org/en/categories/games/
https://f-droid.org/en/categories/graphics/
https://f-droid.org/en/categories/icon-pack/
https://f-droid.org/en/categories/internet/
https://f-droid.org/en/categories/keyboard-ime/
h

In [4]:
PKG_PATH_RE = re.compile(r"^/(?:[a-z]{2}/)?packages/[^/]+(?:/index\.html|/)?$")

def canonicalize_package_url(full_url: str) -> str:
    """把 .../packages/<pkgid>/index.html 归一化为 .../packages/<pkgid>/"""
    p = urlparse(full_url)
    path = p.path
    if path.endswith("/index.html"):
        path = path[:-len("/index.html")] + "/"
    elif not path.endswith("/"):
        path = path + "/"
    return urlunparse((p.scheme, p.netloc, path, "", "", ""))

def extract_package_links_from_listing(soup, page_url: str):
    links = set()
    base_host = urlparse(BASE).netloc
    for a in soup.find_all("a", href=True):
        candidate = urljoin(page_url, a["href"])  # ← 关键修复
        parsed = urlparse(candidate)
        if parsed.netloc == base_host and PKG_PATH_RE.match(parsed.path):
            links.add(canonicalize_package_url(candidate))
    return sorted(links)

def iter_category_pages(category_url):
    base = category_url if category_url.endswith("/") else category_url + "/"
    yield base
    page = 2
    while True:
        yield f"{base}{page}/index.html"
        page += 1

def crawl_category_apps(category_url, verbose=True):
    all_links = []
    for page_url in iter_category_pages(category_url):
        try:
            soup = get_soup(page_url)
        except requests.HTTPError as e:
            if getattr(e.response, "status_code", None) == 404:
                if verbose: print(f"  结束 {page_url} -> 404")
                break
            if verbose: print(f"  停止 {page_url} -> HTTP {e.response.status_code}")
            break
        except Exception as e:
            if verbose: print(f"  访问异常 {page_url}: {e}")
            break

        apps = extract_package_links_from_listing(soup, page_url)  # ← 传 page_url
        if verbose:
            print(f"  {page_url} 抓到 {len(apps)} 个")
        all_links.extend(apps)
        time.sleep(0.2)
    return sorted(set(all_links))


In [ ]:
test_link = "https://f-droid.org/en/categories/games/"
test_apps = crawl_category_apps(test_link, verbose=True)
print("development 分类总数：", len(test_apps))


  https://f-droid.org/en/categories/games/ 抓到 30 个
  https://f-droid.org/en/categories/games/2/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/3/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/4/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/5/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/6/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/7/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/8/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/9/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/10/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/11/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/12/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/13/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/14/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/15/index.html 抓到 30 个
  https://f-droid.org/en/categories/games/16/index.html 抓到 30

In [7]:
category_to_apps = {}
for link in category_links:
    print(f"分类: {link}")
    apps = crawl_category_apps(link, verbose=True)
    category_to_apps[link] = apps

# 去重
app_to_url = {}
for apps in category_to_apps.values():
    for app_url in apps:
        pkg = app_url.rstrip("/").split("/")[-1]
        app_to_url[pkg] = app_url

print(f"\n去重后共 {len(app_to_url)} 个 app")


分类: https://f-droid.org/en/categories/app-store-updater/
  https://f-droid.org/en/categories/app-store-updater/ 抓到 15 个
  结束 https://f-droid.org/en/categories/app-store-updater/2/index.html -> 404
分类: https://f-droid.org/en/categories/bookmark/
  https://f-droid.org/en/categories/bookmark/ 抓到 15 个
  结束 https://f-droid.org/en/categories/bookmark/2/index.html -> 404
分类: https://f-droid.org/en/categories/browser/
  https://f-droid.org/en/categories/browser/ 抓到 15 个
  结束 https://f-droid.org/en/categories/browser/2/index.html -> 404
分类: https://f-droid.org/en/categories/calculator/
  https://f-droid.org/en/categories/calculator/ 抓到 30 个
  https://f-droid.org/en/categories/calculator/2/index.html 抓到 7 个
  结束 https://f-droid.org/en/categories/calculator/3/index.html -> 404
分类: https://f-droid.org/en/categories/calendar-agenda/
  https://f-droid.org/en/categories/calendar-agenda/ 抓到 30 个
  https://f-droid.org/en/categories/calendar-agenda/2/index.html 抓到 10 个
  结束 https://f-droid.org/en/catego

In [13]:
import csv
import time
from urllib.parse import urljoin, urlsplit, urlunsplit

# ALL_OUTPUT = "fdroid_source_links_unique.csv"
# GITHUB_OUTPUT = "fdroid_github_unique.csv"
ALL_OUTPUT = "games_links_unique.csv"
GITHUB_OUTPUT = "games_github_unique.csv"

POLITE_DELAY = 0.35  # 访问详情页节流

def absolutize(href: str) -> str:
    """把相对链接转成绝对；处理 // 开头 和 / 开头的情况"""
    if not href:
        return ""
    href = href.strip()
    if href.startswith("//"):
        return "https:" + href
    if href.startswith("/"):
        return urljoin(BASE, href)
    return href

def normalize_github_repo(url: str) -> str:
    """把各种 GitHub 链接统一到仓库根：https://github.com/<owner>/<repo>"""
    if not url:
        return url
    u = url.strip()
    if u.lower().startswith("http://github.com"):
        u = "https://" + u[7:]
    if not u.lower().startswith("https://github.com"):
        return u
    parts = urlsplit(u)
    comps = [c for c in parts.path.split("/") if c]
    if len(comps) >= 2:
        owner, repo = comps[0], comps[1]
        if repo.endswith(".git"):
            repo = repo[:-4]
        path = f"/{owner}/{repo}"
    elif len(comps) == 1:
        path = f"/{comps[0]}"
    else:
        path = "/"
    return urlunsplit(("https", "github.com", path, "", ""))
def get_source_code_from_app(app_url):
    soup = get_soup(app_url)
    a = soup.find("a", string=lambda s: isinstance(s, str) and s.strip().lower() == "source code")
    if a and a.get("href"):
        return a["href"].strip()
    possible = soup.find(lambda tag: tag.name in ("dt", "th", "h3", "span", "strong")
                         and tag.get_text(strip=True).lower() == "source code")
    if possible and possible.parent:
        link = possible.parent.find("a", href=True)
        if link:
            return link["href"].strip()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if any(host in href for host in ("github.com", "gitlab.com", "codeberg.org", "bitbucket.org", "gitea")):
            return href
    return None

unique_all = set()
unique_github = set()
errors = []

# for i, (pkg, app_url) in enumerate(app_to_url.items(), 1):
for i,  app_url in enumerate(test_apps):
    try:
        src = get_source_code_from_app(app_url)
        src = absolutize(src)
        if src:
            unique_all.add(src)
            if src.lower().startswith(("https://github.com", "http://github.com")):
                unique_github.add(normalize_github_repo(src))
    except Exception as e:
        errors.append( app_url, str(e))
    if i % 100 == 0:
        print(f"进度：{i}/{len(test_apps)}")
    time.sleep(POLITE_DELAY)

print(f"唯一的 source_code_url（全部托管平台）共 {len(unique_all)} 条")
print(f"唯一的 GitHub 仓库链接共 {len(unique_github)} 条")

# 保存“所有平台”的唯一链接
with open(ALL_OUTPUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["source_code_url"])
    for url in sorted(unique_all):
        w.writerow([url])

# 保存“GitHub 仓库级归一化”的唯一链接
with open(GITHUB_OUTPUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["source_code_url"])
    for url in sorted(unique_github):
        w.writerow([url])

print(f"已保存：{ALL_OUTPUT} 与 {GITHUB_OUTPUT}")


进度：0/494
进度：100/494
进度：200/494
进度：300/494
进度：400/494
唯一的 source_code_url（全部托管平台）共 494 条
唯一的 GitHub 仓库链接共 374 条
已保存：games_links_unique.csv 与 games_github_unique.csv


In [ ]:
def get_source_code_from_app(app_url):
    soup = get_soup(app_url)
    a = soup.find("a", string=lambda s: isinstance(s, str) and s.strip().lower() == "source code")
    if a and a.get("href"):
        return a["href"].strip()
    possible = soup.find(lambda tag: tag.name in ("dt", "th", "h3", "span", "strong")
                         and tag.get_text(strip=True).lower() == "source code")
    if possible and possible.parent:
        link = possible.parent.find("a", href=True)
        if link:
            return link["href"].strip()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if any(host in href for host in ("github.com", "gitlab.com", "codeberg.org", "bitbucket.org", "gitea")):
            return href
    return None

# OUTPUT_FILE = "fdroid_source_links_new.csv"
OUTPUT_FILE = "games_app.csv"
POLITE_DELAY = 0.25

results = []
for i, (pkg, app_url) in enumerate(app_to_url.items(), 1):
    try:
        src = get_source_code_from_app(app_url)
    except Exception as e:
        print(f"[{i}] {pkg} 错误: {e}")
        src = None
    results.append([pkg, app_url, src])
    if i % 50 == 0:
        print(f"进度: {i}/{len(app_to_url)}")
    time.sleep(POLITE_DELAY)

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["package_id", "app_url", "source_code_url"])
    w.writerows(results)

print(f"完成！保存 {len(results)} 条到 {OUTPUT_FILE}")


NameError: name 'app_to_url' is not defined

In [14]:
import csv

input_file = OUTPUT_FILE   # 你前面保存的 fdroid_source_links.csv
output_file = "fdroid_github_only.csv"

unique_github = set()

with open(input_file, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        src = (row.get("source_code_url") or "").strip()
        if src.startswith("https://github.com"):
            unique_github.add(src)

print(f"筛选出 {len(unique_github)} 条唯一的 GitHub 链接")

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["source_code_url"])
    for url in sorted(unique_github):
        writer.writerow([url])

print(f"已保存到 {output_file}")


筛选出 2941 条唯一的 GitHub 链接
已保存到 fdroid_github_only.csv
